# BioRAG-X — 04 Advanced Chunking

## Semantic + Biomedical-aware + Proposition + Parent–Child + Late Chunking

Notebook 03 established fixed-size and recursive baselines. This notebook tests richer evidence representations without changing the downstream retriever/generator.

### Strategies implemented
1. Semantic chunking
2. Biomedical-aware / entity-preserving chunking
3. Proposition / atomic-evidence chunking
4. Parent–child hierarchical chunking
5. Late-chunking span representation

**Research rule:** no strategy is declared the winner here. We measure representation quality, coverage, redundancy, and economics first; retrieval impact is evaluated later.

Recent research supports this controlled approach: semantic chunking has not consistently justified its added cost over fixed chunking, while recent biomedical work reports task-dependent gains from entity-preserving and proposition-oriented chunk construction.


In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
import hashlib, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
CANONICAL_DIR = Path("data/canonical")
CHUNK_DIR = Path("data/chunks")
EMBED_DIR = Path("data/embeddings")
ARTIFACT_DIR = Path("artifacts/04_advanced_chunking")
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = CANONICAL_DIR / "passages.parquet"
QUESTIONS_PATH = CANONICAL_DIR / "questions.parquet"
GOLD_REL_PATH = CANONICAL_DIR / "gold_relationships.parquet"

if not PASSAGES_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 first: data/canonical/passages.parquet")

passages = pd.read_parquet(PASSAGES_PATH)
questions = pd.read_parquet(QUESTIONS_PATH) if QUESTIONS_PATH.exists() else None
gold_relationships = pd.read_parquet(GOLD_REL_PATH) if GOLD_REL_PATH.exists() else None

print("Passages:", len(passages))
print("Questions:", None if questions is None else len(questions))
print("Gold relationships:", None if gold_relationships is None else len(gold_relationships))


Passages: 40221
Questions: 4719
Gold relationships: 42608


## Research context

- **Adaptive Chunking: Optimizing Chunking-Method Selection for RAG** proposes selecting chunking strategies per document using intrinsic measures such as References Completeness, Intrachunk Cohesion, Document Contextual Coherence, Block Integrity and Size Compliance. 
- **Is Semantic Chunking Worth the Computational Cost?** reports that semantic chunking does not consistently justify its additional computation over simpler fixed chunking.
- **Configurable Semantic Chunking for Biomedical Information Extraction in RAG** uses entity-preserving windows, trigger-centered chunking, proposition-first extraction and hierarchical relation resolution; its results are explicitly task-dependent.
- **Graph-Aware Late Chunking for Retrieval-Augmented Generation in Biomedical Literature** motivates context-aware chunk representations and structural coverage for biomedical literature.

These findings are why BioRAG-X treats chunking as an empirical retrieval-design problem rather than assuming one universal strategy.


In [2]:
TOKEN_RE = re.compile(r"\S+")

def tokens(text):
    return TOKEN_RE.findall(text or "")

def n_tokens(text):
    return len(tokens(text))

def normalize_spaces(text):
    return re.sub(r"\s+", " ", (text or "").strip())

def sentences(text):
    text = (text or "").strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text) if s.strip()]

def chunk_id(parent_id, strategy, idx, text):
    raw = f"{parent_id}|{strategy}|{idx}|{text}".encode()
    return "CHK-" + hashlib.sha256(raw).hexdigest()[:20]

@dataclass
class ChunkRecord:
    chunk_id: str
    parent_passage_id: str
    strategy: str
    chunk_index: int
    text: str
    approximate_tokens: int
    char_count: int
    start_token: int
    end_token: int
    overlap_tokens: int
    metadata_json: str

def records_to_df(records):
    return pd.DataFrame([asdict(x) for x in records])

def duplicate_ratio(df):
    if len(df) == 0:
        return 0.0
    normalized = df["text"].map(lambda x: normalize_spaces(x).lower())
    return float(normalized.duplicated().mean())


## 1. Semantic chunking

### Idea

Instead of splitting every N tokens, detect **topic/semantic shifts between adjacent sentences**.

Conceptually:

`sentence embeddings → adjacent similarity → boundaries → chunks`

### Trade-off

Semantic chunking requires embedding computation and a boundary threshold. Research shows that the extra cost is not always rewarded by better retrieval, so BioRAG-X records the model, threshold and runtime as part of the experiment.


In [3]:
# Semantic chunking uses Azure OpenAI text-embedding-ada-002 (no Hugging Face / local transformers).
import os, time
from dotenv import load_dotenv
from openai import AzureOpenAI

ENV_PATH = Path("..") / ".env"
load_dotenv(ENV_PATH if ENV_PATH.exists() else None)

EMBEDDING_DEPLOYMENT = os.getenv("embedding_model_name", "text-embedding-ada-002")
SEMANTIC_MODEL_NAME = f"azure-openai:{EMBEDDING_DEPLOYMENT}"

embedding_client = None
semantic_available = False
try:
    embedding_client = AzureOpenAI(
        api_key=os.getenv("embedding_api_key"),
        api_version=os.getenv("embedding_api_version"),
        azure_endpoint=os.getenv("embedding_endpoint_url"),
    )
    _probe = embedding_client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=["probe"])
    EMBEDDING_DIM = len(_probe.data[0].embedding)
    semantic_available = True
except Exception as e:
    print("Azure embedding client unavailable, using deterministic fallback. Reason:", type(e).__name__)
    EMBEDDING_DIM = None

def embed_texts(texts, batch_size=256, max_retries=3):
    """Embed a list of texts with Azure OpenAI ada-002, batched with simple retry."""
    if embedding_client is None:
        raise RuntimeError("Embedding client not available")
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(max_retries):
            try:
                resp = embedding_client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=batch)
                vectors.extend([d.embedding for d in resp.data])
                break
            except Exception:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)
    return np.asarray(vectors, dtype=np.float32)

def adjacent_cosine(vectors):
    """Cosine similarity between consecutive rows of a (n, d) matrix -> length n-1."""
    if len(vectors) < 2:
        return np.array([], dtype=np.float32)
    norm = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12)
    return np.sum(norm[:-1] * norm[1:], axis=1)

print("Semantic embedding backend:", SEMANTIC_MODEL_NAME)
print("Semantic model available:", semantic_available, "| dim:", EMBEDDING_DIM)


Semantic embedding backend: azure-openai:text-embedding-ada-002
Semantic model available: True | dim: 1536


In [4]:
def semantic_chunks(parent_id, text, similarity_threshold=0.72, max_tokens=384, min_tokens=40):
    sents = sentences(text)
    if not sents:
        return []

    if not semantic_available or embedding_client is None:
        # Portable deterministic fallback for notebook development.
        groups, current = [], []
        for s in sents:
            if current and n_tokens(" ".join(current + [s])) > max_tokens:
                groups.append(" ".join(current))
                current = [s]
            else:
                current.append(s)
        if current:
            groups.append(" ".join(current))
    else:
        emb = embed_texts(sents)
        sims = adjacent_cosine(emb)
        groups, current = [], [sents[0]]
        for i, s in enumerate(sents[1:]):
            semantic_break = float(sims[i]) < similarity_threshold
            size_break = n_tokens(" ".join(current + [s])) > max_tokens
            if semantic_break or size_break:
                groups.append(" ".join(current))
                current = [s]
            else:
                current.append(s)
        if current:
            groups.append(" ".join(current))

    merged = []
    for g in groups:
        if merged and n_tokens(g) < min_tokens and n_tokens(merged[-1] + " " + g) <= max_tokens:
            merged[-1] = merged[-1] + " " + g
        else:
            merged.append(g)

    return [
        ChunkRecord(
            chunk_id=chunk_id(parent_id, "semantic", i, g),
            parent_passage_id=parent_id,
            strategy="semantic",
            chunk_index=i,
            text=normalize_spaces(g),
            approximate_tokens=n_tokens(g),
            char_count=len(g),
            start_token=-1,
            end_token=-1,
            overlap_tokens=0,
            metadata_json=json.dumps({
                "similarity_threshold": similarity_threshold,
                "max_tokens": max_tokens,
                "min_tokens": min_tokens,
                "model": SEMANTIC_MODEL_NAME if semantic_available else "rule_based_fallback"
            }, sort_keys=True)
        )
        for i, g in enumerate(merged)
    ]

demo_text = (
    "BRCA1 participates in DNA damage repair. Loss of BRCA1 function increases "
    "homologous recombination deficiency. PARP inhibition can create synthetic "
    "lethality in cells with homologous recombination defects. In another cohort, "
    "response was associated with a separate biomarker."
)
display(pd.DataFrame([asdict(x) for x in semantic_chunks("DEMO", demo_text, max_tokens=35, min_tokens=8)]))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-5307eead10090719fec5,DEMO,semantic,0,BRCA1 participates in DNA damage repair. Loss ...,26,205,-1,-1,0,"{""max_tokens"": 35, ""min_tokens"": 8, ""model"": ""..."
1,CHK-82b98e5f48a402684e56,DEMO,semantic,1,"In another cohort, response was associated wit...",10,69,-1,-1,0,"{""max_tokens"": 35, ""min_tokens"": 8, ""model"": ""..."


## 2. Biomedical-aware chunking

Generic semantic similarity does not explicitly protect biomedical meaning.

We create a deterministic preservation-oriented baseline that identifies:

- gene/protein-like entities
- drug/disease-like entities
- relation triggers
- negation cues

and prefers not to separate relation/negation-bearing statements from their immediate context when the size budget permits.

This is intentionally a **baseline heuristic**, not a production biomedical NER/RE model.


In [5]:
PATTERNS = {
    "gene_like": re.compile(r"\b(?:BRCA\d*|TP\d+|EGFR|KRAS|BRAF|MYC|PTEN|APC)\b", re.I),
    "protein_like": re.compile(r"\b(?:p53|COX[- ]?2|TNF[- ]?alpha|IL[- ]?6|VEGF|HER2)\b", re.I),
    "drug_like": re.compile(r"\b(?:aspirin|ibuprofen|metformin|olaparib|pembrolizumab|trastuzumab|tamoxifen|bevacizumab)\b", re.I),
    "disease_like": re.compile(r"\b(?:breast cancer|ovarian cancer|lung cancer|diabetes|melanoma|alzheimer(?:'s)? disease)\b", re.I),
    "negation": re.compile(r"\b(?:no|not|never|without|lack(?:s|ing)?|absence of|did not|does not)\b", re.I),
    "relation_trigger": re.compile(r"\b(?:inhibits?|activates?|associated with|increases?|decreases?|causes?|prevents?|reduces?|induces?|expressed in|interacts with|sensitizes?|resistant to|responsive to)\b", re.I),
}

def biomedical_signals(text):
    return {
        k: sorted({m.group(0) for m in p.finditer(text)})
        for k, p in PATTERNS.items()
    }

def biomedical_chunks(parent_id, text, max_tokens=384, min_tokens=40):
    sents = sentences(text)
    if not sents:
        return []

    groups, current = [], []
    current_len = 0

    for i, s in enumerate(sents):
        s_len = n_tokens(s)
        sig = biomedical_signals(s)
        relationish = bool(sig["relation_trigger"] or sig["negation"])
        exceeds = bool(current and current_len + s_len > max_tokens)

        if exceeds:
            if relationish and current_len < int(max_tokens * 1.15):
                current.append(s)
                current_len += s_len
                groups.append(" ".join(current))
                current, current_len = [], 0
            else:
                groups.append(" ".join(current))
                current, current_len = [s], s_len
        else:
            current.append(s)
            current_len += s_len

    if current:
        groups.append(" ".join(current))

    return [
        ChunkRecord(
            chunk_id=chunk_id(parent_id, "biomedical", i, g),
            parent_passage_id=parent_id,
            strategy="biomedical",
            chunk_index=i,
            text=normalize_spaces(g),
            approximate_tokens=n_tokens(g),
            char_count=len(g),
            start_token=-1,
            end_token=-1,
            overlap_tokens=0,
            metadata_json=json.dumps({
                "signals": biomedical_signals(g),
                "max_tokens": max_tokens,
                "min_tokens": min_tokens
            }, sort_keys=True)
        )
        for i, g in enumerate(groups)
    ]

bio_demo = biomedical_chunks(
    "DEMO",
    "BRCA1 mutations were associated with homologous recombination deficiency. "
    "This deficiency increases sensitivity to PARP inhibition. However, one cohort "
    "did not show a significant treatment response.",
    max_tokens=35,
    min_tokens=8
)
display(pd.DataFrame([asdict(x) for x in bio_demo]))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-09837b2a9d0b68ba336f,DEMO,biomedical,0,BRCA1 mutations were associated with homologou...,25,198,-1,-1,0,"{""max_tokens"": 35, ""min_tokens"": 8, ""signals"":..."


## 3. Proposition / atomic-evidence chunking

### Idea

Represent one relatively self-contained claim as the retrieval unit.

Example:

`BRCA1 mutations increase homologous recombination deficiency.`

### Benefit

Fine-grained retrieval can improve evidence localization.

### Risk

Too much atomicity can remove context.

Therefore every proposition retains its `parent_passage_id`; later evidence selection can expand proposition → parent context.

This notebook uses a conservative rule-based approximation. A future production version may use a biomedical proposition/IE model.


In [6]:
def proposition_candidates(sentence):
    parts = re.split(
        r"\s+(?:and|but|while|whereas|although|because|therefore|however)\s+",
        sentence, flags=re.I
    )
    return [p.strip(" ,;:") for p in parts if p.strip()] or [sentence]

def proposition_chunks(parent_id, text, max_tokens=96):
    props = []
    for s in sentences(text):
        for p in proposition_candidates(s):
            toks = tokens(p)
            if len(toks) <= max_tokens:
                props.append(p)
            else:
                props.extend(
                    " ".join(toks[i:i+max_tokens])
                    for i in range(0, len(toks), max_tokens)
                )

    return [
        ChunkRecord(
            chunk_id=chunk_id(parent_id, "proposition", i, p),
            parent_passage_id=parent_id,
            strategy="proposition",
            chunk_index=i,
            text=normalize_spaces(p),
            approximate_tokens=n_tokens(p),
            char_count=len(p),
            start_token=-1,
            end_token=-1,
            overlap_tokens=0,
            metadata_json=json.dumps({
                "granularity": "atomic_evidence_approximation",
                "signals": biomedical_signals(p)
            }, sort_keys=True)
        )
        for i, p in enumerate(props)
    ]

prop_demo = proposition_chunks(
    "DEMO",
    "BRCA1 mutations increase homologous recombination deficiency and PARP inhibition "
    "can create synthetic lethality in affected cells."
)
display(pd.DataFrame([asdict(x) for x in prop_demo]))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-65b1f79a9000b2953c2b,DEMO,proposition,0,BRCA1 mutations increase homologous recombinat...,6,60,-1,-1,0,"{""granularity"": ""atomic_evidence_approximation..."
1,CHK-c012c64f759af459b320,DEMO,proposition,1,PARP inhibition can create synthetic lethality...,9,65,-1,-1,0,"{""granularity"": ""atomic_evidence_approximation..."


## 4. Parent–child hierarchical chunking

Parent–child separates:

**retrieval granularity** from **generation context**.

```text
Parent passage
   ├── child 1
   ├── child 2
   ├── child 3
   └── child 4
```

Search children; when evidence is selected, recover the parent or a controlled local context window.

Every child must preserve `parent_passage_id`.


In [7]:
def parent_child_chunks(parent_id, text, child_size=96, child_overlap=16):
    toks = tokens(text)
    if not toks:
        return []
    if child_overlap >= child_size:
        raise ValueError("child_overlap must be smaller than child_size")

    records = []
    step = child_size - child_overlap
    start, idx = 0, 0

    while start < len(toks):
        end = min(start + child_size, len(toks))
        child_text = " ".join(toks[start:end])
        records.append(ChunkRecord(
            chunk_id=chunk_id(parent_id, "parent_child", idx, child_text),
            parent_passage_id=parent_id,
            strategy="parent_child",
            chunk_index=idx,
            text=child_text,
            approximate_tokens=end-start,
            char_count=len(child_text),
            start_token=start,
            end_token=end,
            overlap_tokens=0 if idx == 0 else child_overlap,
            metadata_json=json.dumps({
                "retrieval_unit": "child",
                "generation_context": "parent",
                "child_size": child_size,
                "child_overlap": child_overlap
            }, sort_keys=True)
        ))
        idx += 1
        if end == len(toks):
            break
        start += step

    return records

display(pd.DataFrame([asdict(x) for x in parent_child_chunks(
    "DEMO",
    "Sentence one. Sentence two contains evidence. Sentence three provides additional context. Sentence four concludes.",
    child_size=10,
    child_overlap=2
)]))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-cf1e07669b6ceb7d69b5,DEMO,parent_child,0,Sentence one. Sentence two contains evidence. ...,10,80,0,10,0,"{""child_overlap"": 2, ""child_size"": 10, ""genera..."
1,CHK-31ffb2a4a30008e90483,DEMO,parent_child,1,provides additional context. Sentence four con...,6,53,8,14,2,"{""child_overlap"": 2, ""child_size"": 10, ""genera..."


## 5. Late chunking

### Conventional

`document → chunks → independent embeddings`

### Late

`document/long context → contextual token embeddings → span pooling → chunk embeddings`

Late chunking is an **embedding representation strategy**, not just another text splitter.

Therefore this notebook creates the span contract. Actual model embeddings are deferred to Notebook 06, where exact dense and ANN retrieval will be measured.

We will never label ordinary chunk embeddings as "late chunking".


In [8]:
def late_chunk_spans(text, chunk_size=256, overlap=32):
    toks = tokens(text)
    if not toks:
        return []
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    spans = []
    step = chunk_size - overlap
    start, idx = 0, 0
    while start < len(toks):
        end = min(start + chunk_size, len(toks))
        spans.append({
            "chunk_index": idx,
            "start_token": start,
            "end_token": end,
            "text": " ".join(toks[start:end]),
            "approximate_tokens": end-start,
        })
        idx += 1
        if end == len(toks):
            break
        start += step
    return spans

display(pd.DataFrame(late_chunk_spans(
    "Sentence one. Sentence two contains context. Sentence three contains a relation.",
    chunk_size=10,
    overlap=2
)))


,chunk_index,start_token,end_token,text,approximate_tokens
0,0,0,10,Sentence one. Sentence two contains context. S...,10
1,1,8,11,contains a relation.,3


## 6. Run the advanced strategies on a reproducible sample

For notebook development we default to a bounded sample. Set `ADVANCED_SAMPLE_N = None` for full-corpus processing after verifying the implementation.

The sample size is not an evaluation set.


In [9]:
ADVANCED_SAMPLE_N = min(5000, len(passages))
advanced_sample = (
    passages.sample(n=ADVANCED_SAMPLE_N, random_state=SEED)
    if ADVANCED_SAMPLE_N else passages.copy()
)

print("Sample:", len(advanced_sample))


Sample: 5000


In [10]:
def apply_strategy(df, name, fn):
    rows = []
    for row in df[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
        records = fn(row.canonical_passage_id, row.retrieval_text)
        rows.extend(asdict(r) for r in records)
    return pd.DataFrame(rows)

semantic_df = apply_strategy(
    advanced_sample, "semantic",
    lambda pid, text: semantic_chunks(pid, text, 0.72, 384, 40)
)
biomedical_df = apply_strategy(
    advanced_sample, "biomedical",
    lambda pid, text: biomedical_chunks(pid, text, 384, 40)
)
proposition_df = apply_strategy(
    advanced_sample, "proposition",
    lambda pid, text: proposition_chunks(pid, text, 96)
)
parent_child_df = apply_strategy(
    advanced_sample, "parent_child",
    lambda pid, text: parent_child_chunks(pid, text, 96, 16)
)

late_rows = []
for row in advanced_sample[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
    for span in late_chunk_spans(row.retrieval_text, 256, 32):
        text = span["text"]
        late_rows.append({
            "chunk_id": chunk_id(row.canonical_passage_id, "late", span["chunk_index"], text),
            "parent_passage_id": row.canonical_passage_id,
            "strategy": "late",
            "chunk_index": span["chunk_index"],
            "text": text,
            "approximate_tokens": span["approximate_tokens"],
            "char_count": len(text),
            "start_token": span["start_token"],
            "end_token": span["end_token"],
            "overlap_tokens": 0 if span["chunk_index"] == 0 else 32,
            "metadata_json": json.dumps({
                "embedding_status": "deferred_to_notebook_06",
                "representation": "late_chunking_span"
            }, sort_keys=True)
        })
late_df = pd.DataFrame(late_rows)

print("Semantic:", len(semantic_df))
print("Biomedical:", len(biomedical_df))
print("Proposition:", len(proposition_df))
print("Parent-child:", len(parent_child_df))
print("Late spans:", len(late_df))


Semantic: 3610
Biomedical: 3562
Proposition: 59536
Parent-child: 10096
Late spans: 4238


## 7. Compare representation economics

Important production metrics:

- chunks created
- chunks per source passage
- median/p95 chunk size
- duplicate text ratio
- storage/index expansion

A strategy that improves recall but triples the index and reranking cost may not be the production winner.


In [11]:
def representation_stats(name, df, source_count):
    if df.empty:
        return {"strategy": name, "chunks": 0}
    return {
        "strategy": name,
        "chunks": len(df),
        "source_passages": source_count,
        "chunks_per_passage": len(df) / max(1, source_count),
        "median_tokens": float(df["approximate_tokens"].median()),
        "p95_tokens": float(df["approximate_tokens"].quantile(.95)),
        "duplicate_text_ratio": duplicate_ratio(df),
        "empty_chunk_rate": float((df["approximate_tokens"] == 0).mean()),
    }

economics = pd.DataFrame([
    representation_stats("semantic", semantic_df, len(advanced_sample)),
    representation_stats("biomedical", biomedical_df, len(advanced_sample)),
    representation_stats("proposition", proposition_df, len(advanced_sample)),
    representation_stats("parent_child", parent_child_df, len(advanced_sample)),
    representation_stats("late", late_df, len(advanced_sample)),
])

display(economics)


,strategy,chunks,source_passages,chunks_per_passage,median_tokens,p95_tokens,duplicate_text_ratio,empty_chunk_rate
0,semantic,3610,5000,0.7220,201.0,325.55,0.000554,0.0
1,biomedical,3562,5000,0.7124,202.0,327.00,0.000561,0.0
2,proposition,59536,5000,11.9072,10.0,27.00,0.017183,0.0
3,parent_child,10096,5000,2.0192,96.0,96.00,0.000198,0.0
4,late,4238,5000,0.8476,192.0,256.00,0.000472,0.0


## 7b. Tokenizer reality check

Every size number in this notebook counts **whitespace words** (`\\S+`).
The embedding model that will index these chunks in Notebook 06 counts
**sub-word model tokens**, which is a larger number. If we plan chunk budgets
in whitespace words, our real token budgets are silently smaller than intended.

We calibrate the ratio on a sample of the actual chunks. If `tiktoken` is
available we use `cl100k_base` (the ada-002 / modern OpenAI tokenizer);
otherwise we fall back to a documented heuristic. This mirrors Notebook 03's
tokenizer sanity check and keeps every size claim in this notebook honest.


In [12]:
# Whitespace-word vs real model-token calibration.
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def real_tokens(text):
        return len(_enc.encode(text or ""))
    TOKENIZER_BACKEND = "tiktoken:cl100k_base"
except Exception as e:
    _enc = None
    def real_tokens(text):
        # Documented fallback: ~1.3 sub-word tokens per whitespace word for
        # dense biomedical prose. Only used if tiktoken is unavailable.
        return int(round(n_tokens(text) * 1.3))
    TOKENIZER_BACKEND = f"fallback_1.3x ({type(e).__name__})"

# Calibrate on a reproducible sample drawn across all built strategies.
_tok_frames = {
    "semantic": semantic_df, "biomedical": biomedical_df,
    "proposition": proposition_df, "parent_child": parent_child_df, "late": late_df,
}
_tok_rows = []
for _name, _df in _tok_frames.items():
    if _df.empty:
        continue
    _s = _df["text"].sample(n=min(400, len(_df)), random_state=SEED)
    _ws = _s.map(n_tokens)
    _rt = _s.map(real_tokens)
    _ratio = (_rt / _ws.replace(0, np.nan)).dropna()
    _tok_rows.append({
        "strategy": _name,
        "sampled_chunks": int(len(_s)),
        "median_whitespace_tokens": float(_ws.median()),
        "median_real_tokens": float(_rt.median()),
        "mean_real_per_word": float(_ratio.mean()),
        "p95_real_per_word": float(_ratio.quantile(.95)),
    })

tokenizer_reality = pd.DataFrame(_tok_rows)
_overall_ratio = float(tokenizer_reality["mean_real_per_word"].mean()) if not tokenizer_reality.empty else float("nan")

print("Tokenizer backend:", TOKENIZER_BACKEND)
print(f"Overall real-tokens-per-whitespace-word: {_overall_ratio:.3f}x")
print(f"=> a nominal 384-word budget is ~{int(round(384 * _overall_ratio))} real model tokens.")
display(tokenizer_reality)

tokenizer_reality.to_csv(ARTIFACT_DIR / "tokenizer_reality.csv", index=False)


Tokenizer backend: tiktoken:cl100k_base
Overall real-tokens-per-whitespace-word: 1.566x
=> a nominal 384-word budget is ~601 real model tokens.


,strategy,sampled_chunks,median_whitespace_tokens,median_real_tokens,mean_real_per_word,p95_real_per_word
0,semantic,400,199.0,295.5,1.520545,1.847884
1,biomedical,400,202.5,305.0,1.518644,1.863894
2,proposition,400,10.0,15.0,1.748311,3.000000
3,parent_child,400,96.0,133.0,1.521651,1.948779
4,late,400,190.5,279.0,1.522109,1.842972


## 7c. Chunk-explosion and index-cost economics

Section 7 reports *shape* (chunks, size, duplication). It does not report the
**downstream cost** of that shape. A strategy that improves localization but
triples the vectors to embed, store, rerank and search may not be a production
winner.

The proposition strategy above produces ~12 chunks per source passage — an
order of magnitude more than the others. Here we quantify what that explosion
costs: total chunks, total real tokens to embed, an embedding-cost proxy
(ada-002 is billed per token), and the index-expansion factor relative to the
semantic representation. These are cost measurements only; no winner is chosen.


In [13]:
# Index-cost economics per strategy (uses the real-token calibration from 7b).
# Public ada-002 list price at time of writing; adjust if your contract differs.
ADA_002_USD_PER_1K_TOKENS = 0.0001

def _index_cost_row(name, df):
    if df.empty:
        return {"strategy": name, "chunks": 0}
    real_tok = df["text"].map(real_tokens)
    total_real = int(real_tok.sum())
    return {
        "strategy": name,
        "chunks": int(len(df)),
        "chunks_per_passage": round(len(df) / max(1, len(advanced_sample)), 3),
        "total_real_tokens_to_embed": total_real,
        "median_real_tokens_per_chunk": float(real_tok.median()),
        "embed_cost_usd_proxy": round(total_real / 1000 * ADA_002_USD_PER_1K_TOKENS, 4),
        "approx_index_bytes_f32_1536d": int(len(df) * 1536 * 4),
    }

_cost_frames = {
    "semantic": semantic_df, "biomedical": biomedical_df,
    "proposition": proposition_df, "parent_child": parent_child_df, "late": late_df,
}
index_economics = pd.DataFrame([_index_cost_row(n, d) for n, d in _cost_frames.items()])

# Index-expansion factor relative to the semantic representation (a compact baseline).
_base = index_economics.loc[index_economics["strategy"] == "semantic", "chunks"]
_base_chunks = int(_base.iloc[0]) if len(_base) and _base.iloc[0] else 1
index_economics["chunk_explosion_vs_semantic"] = (index_economics["chunks"] / _base_chunks).round(2)

display(index_economics)
index_economics.to_csv(ARTIFACT_DIR / "index_economics.csv", index=False)

_worst = index_economics.loc[index_economics["chunks"].idxmax()]
print(f"Largest index: {_worst['strategy']} "
      f"({int(_worst['chunks'])} chunks, "
      f"{_worst['chunk_explosion_vs_semantic']}x the semantic representation, "
      f"~${_worst['embed_cost_usd_proxy']} to embed the sample).")


,strategy,chunks,chunks_per_passage,total_real_tokens_to_embed,median_real_tokens_per_chunk,embed_cost_usd_proxy,approx_index_bytes_f32_1536d,chunk_explosion_vs_semantic
0,semantic,3610,0.722,1115184,300.0,0.1115,22179840,1.00
1,biomedical,3562,0.712,1115180,303.0,0.1115,21884928,0.99
2,proposition,59536,11.907,1092108,15.0,0.1092,365789184,16.49
3,parent_child,10096,2.019,1279108,135.0,0.1279,62029824,2.80
4,late,4238,0.848,1152882,281.5,0.1153,26038272,1.17


Largest index: proposition (59536 chunks, 16.49x the semantic representation, ~$0.1092 to embed the sample).


## 8. Biomedical integrity diagnostics

These are heuristic indicators for now:

- biomedical entity signal count
- relation-bearing chunks
- negation-bearing chunks

A production version should plug in validated biomedical NER/relation extraction and compute true entity/relation integrity against source spans.


In [14]:
def biomedical_stats(df):
    if df.empty:
        return {"entity_signal_count": 0, "relation_chunks": 0, "negation_chunks": 0}
    entity_count = relation_count = negation_count = 0
    for text in df["text"]:
        s = biomedical_signals(text)
        entity_count += sum(len(s[k]) for k in ["gene_like","protein_like","drug_like","disease_like"])
        relation_count += int(bool(s["relation_trigger"]))
        negation_count += int(bool(s["negation"]))
    return {
        "entity_signal_count": entity_count,
        "relation_chunks": relation_count,
        "negation_chunks": negation_count,
    }

integrity = pd.DataFrame({
    "semantic": biomedical_stats(semantic_df),
    "biomedical": biomedical_stats(biomedical_df),
    "proposition": biomedical_stats(proposition_df),
    "parent_child": biomedical_stats(parent_child_df),
    "late": biomedical_stats(late_df),
}).T

display(integrity)


,entity_signal_count,relation_chunks,negation_chunks
semantic,748,1745,1793
biomedical,745,1738,1787
proposition,1818,2944,3142
parent_child,1268,2601,2832
late,813,1882,1968


## 8b. Biomedical integrity ratios (split detection vs source)

Section 8 counts how many *chunks* contain an entity / relation / negation
signal. That is a coverage count, not an integrity measure — it cannot tell us
whether a chunk boundary **fragmented** a biomedical statement.

The failure mode this project cares about is exactly that fragmentation:

```text
"Aspirin inhibits cyclooxygenase"   ->   [Aspirin inhibits] | [cyclooxygenase]
```

Here we measure true **integrity ratios** against the source passage. For each
source passage we detect entity, relation-trigger and negation signals in the
original text, then check whether each signal survives *intact inside a single
chunk* of that passage (rather than being split across a boundary). This is
distinct from Notebook 05's routing features, which score signal presence for
routing rather than preservation against the source.


In [15]:
# True integrity ratios: does each source signal survive intact within ONE chunk?
# We reuse the PATTERNS defined for biomedical signals (entities/relation/negation).
_ENTITY_KEYS = ["gene_like", "protein_like", "drug_like", "disease_like"]

def _signal_spans(text):
    """Return {category: [matched_surface_strings]} using the shared PATTERNS."""
    out = {}
    for k, p in PATTERNS.items():
        out[k] = [m.group(0) for m in p.finditer(text or "")]
    return out

def integrity_ratios(source_df, chunks_df):
    """Fraction of source signals that appear intact inside at least one chunk."""
    by_parent = {pid: list(grp["text"]) for pid, grp in chunks_df.groupby("parent_passage_id")}

    ent_tot = ent_ok = rel_tot = rel_ok = neg_tot = neg_ok = 0
    # Relation integrity is checked on the relation-bearing sentence: the trigger
    # AND its surrounding sentence must stay in one chunk (not split apart).
    for row in source_df[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
        chunks = by_parent.get(row.canonical_passage_id, [])
        if not chunks:
            continue
        sig = _signal_spans(row.retrieval_text)

        # Entity integrity: each entity surface form present intact in some chunk.
        entities = []
        for k in _ENTITY_KEYS:
            entities.extend(sig[k])
        for ent in entities:
            ent_tot += 1
            if any(ent.lower() in c.lower() for c in chunks):
                ent_ok += 1

        # Negation integrity: each negation cue present intact in some chunk.
        for neg in sig["negation"]:
            neg_tot += 1
            if any(neg.lower() in c.lower() for c in chunks):
                neg_ok += 1

        # Relation integrity: for each relation-bearing SOURCE sentence, the whole
        # sentence must fit inside a single chunk (trigger not split from context).
        for sent in sentences(row.retrieval_text):
            if PATTERNS["relation_trigger"].search(sent):
                rel_tot += 1
                norm_sent = normalize_spaces(sent).lower()
                if any(norm_sent in normalize_spaces(c).lower() for c in chunks):
                    rel_ok += 1

    return {
        "entity_integrity": ent_ok / max(1, ent_tot),
        "relation_integrity": rel_ok / max(1, rel_tot),
        "negation_integrity": neg_ok / max(1, neg_tot),
        "entities_checked": ent_tot,
        "relations_checked": rel_tot,
        "negations_checked": neg_tot,
    }

_integ_frames = {
    "semantic": semantic_df, "biomedical": biomedical_df,
    "proposition": proposition_df, "parent_child": parent_child_df, "late": late_df,
}
integrity_ratio_df = pd.DataFrame(
    {name: integrity_ratios(advanced_sample, df) for name, df in _integ_frames.items()}
).T

display(integrity_ratio_df)
integrity_ratio_df.to_csv(ARTIFACT_DIR / "biomedical_integrity_ratios.csv")

print("Interpretation: lower relation_integrity => the strategy splits relation")
print("statements across chunk boundaries more often (evidence fragmentation).")


,entity_integrity,relation_integrity,negation_integrity,entities_checked,relations_checked,negations_checked
semantic,1.0,1.000000,1.0,1878.0,2724.0,3212.0
biomedical,1.0,1.000000,1.0,1878.0,2724.0,3212.0
proposition,1.0,0.352056,1.0,1878.0,2724.0,3212.0
parent_child,1.0,0.906388,1.0,1878.0,2724.0,3212.0
late,1.0,0.997063,1.0,1878.0,2724.0,3212.0


Interpretation: lower relation_integrity => the strategy splits relation
statements across chunk boundaries more often (evidence fragmentation).


## 9. Source-content preservation

For each strategy, compare source-token coverage against all chunks generated from that source passage.

This detects accidental content loss.

Because overlap creates duplication, the target is **coverage**, not exact length equality.


In [16]:
def token_coverage(source_df, chunks_df):
    grouped = chunks_df.groupby("parent_passage_id")["text"].apply(lambda s: " ".join(s))
    rows = []
    for row in source_df[["canonical_passage_id","retrieval_text"]].itertuples(index=False):
        source = tokens(row.retrieval_text.lower())
        recon = tokens(grouped.get(row.canonical_passage_id, "").lower())
        src = Counter(source)
        rec = Counter(recon)
        covered = sum(min(v, rec.get(k, 0)) for k, v in src.items())
        rows.append({
            "parent_passage_id": row.canonical_passage_id,
            "coverage": covered / max(1, len(source))
        })
    return pd.DataFrame(rows)

coverage = {}
for name, df in {
    "semantic": semantic_df,
    "biomedical": biomedical_df,
    "proposition": proposition_df,
    "parent_child": parent_child_df,
    "late": late_df,
}.items():
    c = token_coverage(advanced_sample, df)
    coverage[name] = {
        "median": float(c["coverage"].median()),
        "p05": float(c["coverage"].quantile(.05)),
        "min": float(c["coverage"].min()),
    }

coverage_df = pd.DataFrame(coverage).T
display(coverage_df)


,median,p05,min
semantic,1.000000,0.0,0.0
biomedical,1.000000,0.0,0.0
proposition,0.944444,0.0,0.0
parent_child,1.000000,0.0,0.0
late,1.000000,0.0,0.0


## 9b. Semantic-chunking cost accounting

The research question behind semantic chunking is explicitly *"is it worth the
computational cost?"*. Semantic chunking is the only strategy here that spends
embedding calls to **detect boundaries** (via ada-002). We record what that cost
actually was on the sample so the benefit measured elsewhere can be judged
against it.

If the Azure embedding client was unavailable during the run, semantic chunking
used the deterministic fallback and the embedding cost is zero — which we report
honestly rather than hiding.


In [17]:
# Instrumented re-run of semantic boundary detection on a bounded sub-sample,
# purely to measure cost (calls / tokens / latency / spend proxy). We do NOT
# rebuild the saved chunks here; this is a measurement of the embedding work
# semantic chunking requires.
import time as _time

SEM_COST_SAMPLE_N = min(300, len(advanced_sample))
_sem_cost_sample = advanced_sample.sample(n=SEM_COST_SAMPLE_N, random_state=SEED)

_embed_calls = 0
_sentences_embedded = 0
_tokens_embedded = 0
_t0 = _time.perf_counter()

if semantic_available and embedding_client is not None:
    for _txt in _sem_cost_sample["retrieval_text"]:
        _sents = sentences(_txt)
        if len(_sents) < 2:
            continue
        # Mirror semantic_chunks(): one embedding batch per passage's sentences.
        _ = embed_texts(_sents)
        _embed_calls += 1
        _sentences_embedded += len(_sents)
        _tokens_embedded += sum(real_tokens(s) for s in _sents)
    _elapsed = _time.perf_counter() - _t0
    _backend = SEMANTIC_MODEL_NAME
else:
    _elapsed = 0.0
    _backend = "rule_based_fallback (no embedding cost)"

semantic_cost = {
    "backend": _backend,
    "passages_sampled": int(SEM_COST_SAMPLE_N),
    "embedding_calls": int(_embed_calls),
    "sentences_embedded": int(_sentences_embedded),
    "real_tokens_embedded": int(_tokens_embedded),
    "wall_clock_seconds": round(_elapsed, 3),
    "seconds_per_passage": round(_elapsed / max(1, SEM_COST_SAMPLE_N), 4),
    "embed_cost_usd_proxy": round(_tokens_embedded / 1000 * ADA_002_USD_PER_1K_TOKENS, 6),
}
# Extrapolate the boundary-detection embedding cost to the full sample.
semantic_cost["projected_cost_full_sample_usd"] = round(
    semantic_cost["embed_cost_usd_proxy"] * (len(advanced_sample) / max(1, SEM_COST_SAMPLE_N)), 4
)

semantic_cost_df = pd.DataFrame([semantic_cost])
display(semantic_cost_df)
semantic_cost_df.to_csv(ARTIFACT_DIR / "semantic_cost_accounting.csv", index=False)

print("Note: this is the cost to DETECT boundaries. The other four strategies")
print("spend zero embedding calls at chunking time (they are deterministic).")


,backend,passages_sampled,embedding_calls,sentences_embedded,real_tokens_embedded,wall_clock_seconds,seconds_per_passage,embed_cost_usd_proxy,projected_cost_full_sample_usd
0,azure-openai:text-embedding-ada-002,300,201,1885,70291,93.763,0.3125,0.007029,0.1172


Note: this is the cost to DETECT boundaries. The other four strategies
spend zero embedding calls at chunking time (they are deterministic).


## 10. Parent–child integrity

Parent-child is successful only if every child maps cleanly back to exactly one canonical source passage.

We verify:

`child_id → parent_passage_id`

and later the retrieval layer can expand:

`child → parent → local context`.


In [18]:
pc_parent_counts = parent_child_df.groupby("parent_passage_id").size()

pc_summary = pd.Series({
    "parents_represented": int(parent_child_df["parent_passage_id"].nunique()),
    "children": int(len(parent_child_df)),
    "children_per_parent_median": float(pc_parent_counts.median()),
    "children_per_parent_p95": float(pc_parent_counts.quantile(.95)),
    "orphan_children": int(
        (~parent_child_df["parent_passage_id"].isin(set(advanced_sample["canonical_passage_id"]))).sum()
    ),
})
display(pc_summary.to_frame("value"))


,value
parents_represented,3483.0
children,10096.0
children_per_parent_median,3.0
children_per_parent_p95,4.0
orphan_children,0.0


## 11. Late-chunking implementation contract

The late-chunking dataframe is intentionally **not claiming to contain late embeddings**.

It stores:

- span boundaries
- parent passage ID
- chunk text
- representation type
- embedding status

Notebook 06 will plug a long-context embedding model into these spans and compare:

- ordinary chunk embeddings
- late chunk embeddings
- exact dense retrieval
- ANN retrieval


## 11c. Gold-evidence proxy scorecard (cheap answer-recall, no retriever)

This is the measurement the notebook most needs. Notebook 04 loads the gold
questions and relationships but, until now, only counted them. Following the
same discipline as Notebook 03 (section 11b), we compute a **cheap, retriever-free
proxy** of whether each chunking strategy keeps the answer evidence intact.

For every question that has usable gold evidence, we take its gold passage(s),
find the **single best chunk** of that passage under each strategy, and measure
how much of the answer's content vocabulary that one chunk still contains. A
strategy that fragments evidence forces the answer content across multiple
chunks, lowering the best-single-chunk recall.

This is explicitly a **proxy, not a retrieval benchmark** — the real Recall@K /
MRR verdict is deferred to Notebook 06/07. No winner is declared here.


In [19]:
# Best-single-chunk answer content-token recall per strategy (proxy only).
_STOP = set("the a an of and or to in for with on by is are was were be as at from that this "
            "these those we our it its their his her can may using used based between within into "
            "which who whom whose than then also not no such via per over under more most less".split())
_WORD_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9\-]+")

def content_tokens(text):
    toks = [t.lower() for t in _WORD_RE.findall(text or "")]
    return [t for t in toks if t not in _STOP and len(t) > 2]

def best_single_chunk_recall(chunks_df):
    if chunks_df.empty or questions is None:
        return pd.Series([], dtype=float)
    by_parent = {}
    for pid, grp in chunks_df.groupby("parent_passage_id"):
        by_parent[pid] = [Counter(content_tokens(t)) for t in grp["text"]]

    usable = questions[questions["has_usable_gold_evidence"] == True]
    recalls = []
    for row in usable[["answer", "usable_gold_canonical_ids"]].itertuples(index=False):
        ans = Counter(content_tokens(row.answer))
        if not ans:
            continue
        total = sum(ans.values())
        gold_ids = row.usable_gold_canonical_ids
        gold_ids = list(gold_ids) if gold_ids is not None else []
        best = 0.0
        for gid in gold_ids:
            for ch in by_parent.get(gid, []):
                covered = sum(min(n, ch.get(tok, 0)) for tok, n in ans.items())
                r = covered / total
                if r > best:
                    best = r
            if best >= 0.999:
                break
        recalls.append(best)
    return pd.Series(recalls, dtype=float)

_proxy_frames = {
    "semantic": semantic_df, "biomedical": biomedical_df,
    "proposition": proposition_df, "parent_child": parent_child_df, "late": late_df,
}
_proxy = {name: best_single_chunk_recall(df) for name, df in _proxy_frames.items()}

gold_proxy = pd.DataFrame({
    name: s.describe() for name, s in _proxy.items() if len(s)
}).T
_n_eval = max((len(s) for s in _proxy.values()), default=0)
print("Questions evaluated (usable gold present):", _n_eval)
for name, s in _proxy.items():
    if len(s):
        print(f"  {name:<13} mean best-single-chunk answer recall: {s.mean():.4f}")
display(gold_proxy)

gold_proxy.to_csv(ARTIFACT_DIR / "gold_evidence_proxy.csv")
print("\nProxy only, NOT a retrieval benchmark. Real verdict deferred to NB06/07.")


Questions evaluated (usable gold present): 4385
  semantic      mean best-single-chunk answer recall: 0.2332
  biomedical    mean best-single-chunk answer recall: 0.2335
  proposition   mean best-single-chunk answer recall: 0.1499
  parent_child  mean best-single-chunk answer recall: 0.2116
  late          mean best-single-chunk answer recall: 0.2321


,count,mean,std,min,25%,50%,75%,max
semantic,4385.0,0.233213,0.303321,0.0,0.0,0.0,0.437500,1.0
biomedical,4385.0,0.233504,0.303588,0.0,0.0,0.0,0.437500,1.0
proposition,4385.0,0.149919,0.224727,0.0,0.0,0.0,0.238095,1.0
parent_child,4385.0,0.211597,0.285600,0.0,0.0,0.0,0.375000,1.0
late,4385.0,0.232128,0.302294,0.0,0.0,0.0,0.428571,1.0



Proxy only, NOT a retrieval benchmark. Real verdict deferred to NB06/07.


## 12. Save advanced representations and manifest


In [20]:
outputs = {
    "semantic": CHUNK_DIR / "semantic_advanced.parquet",
    "biomedical": CHUNK_DIR / "biomedical_advanced.parquet",
    "proposition": CHUNK_DIR / "proposition_advanced.parquet",
    "parent_child": CHUNK_DIR / "parent_child_advanced.parquet",
    "late": CHUNK_DIR / "late_spans_advanced.parquet",
}

frames = {
    "semantic": semantic_df,
    "biomedical": biomedical_df,
    "proposition": proposition_df,
    "parent_child": parent_child_df,
    "late": late_df,
}

for name, path in outputs.items():
    frames[name].to_parquet(path, index=False)

manifest = {
    "notebook": "04_advanced_chunking",
    "seed": SEED,
    "sample_size": len(advanced_sample),
    "strategies": {
        "semantic": {
            "threshold": 0.72,
            "max_tokens": 384,
            "min_tokens": 40,
            "embedding_model": SEMANTIC_MODEL_NAME,
            "true_embedding_model_available": semantic_available,
            "chunk_embeddings": str(EMBED_DIR / "semantic_chunk_embeddings.parquet"),
        },
        "biomedical": {
            "method": "deterministic entity/relation/negation preservation heuristic",
            "max_tokens": 384,
            "min_tokens": 40,
        },
        "proposition": {
            "method": "conservative sentence/coordination splitting",
            "max_tokens": 96,
        },
        "parent_child": {
            "child_size": 96,
            "child_overlap": 16,
        },
        "late": {
            "span_size": 256,
            "overlap": 32,
            "embedding_status": "deferred_to_notebook_06",
        },
    },
    "measurements_added": {
        "tokenizer_reality": str(ARTIFACT_DIR / "tokenizer_reality.csv"),
        "index_economics": str(ARTIFACT_DIR / "index_economics.csv"),
        "biomedical_integrity_ratios": str(ARTIFACT_DIR / "biomedical_integrity_ratios.csv"),
        "semantic_cost_accounting": str(ARTIFACT_DIR / "semantic_cost_accounting.csv"),
        "gold_evidence_proxy": str(ARTIFACT_DIR / "gold_evidence_proxy.csv"),
    },
    "retrieval_evaluation_completed": False,
    "winner_selected": False,
    "outputs": {k: str(v) for k, v in outputs.items()},
}

(ARTIFACT_DIR / "advanced_chunking_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Saved advanced chunking artifacts.")
for k, v in outputs.items():
    print(k, "->", v)


Saved advanced chunking artifacts.
semantic -> data\chunks\semantic_advanced.parquet
biomedical -> data\chunks\biomedical_advanced.parquet
proposition -> data\chunks\proposition_advanced.parquet
parent_child -> data\chunks\parent_child_advanced.parquet
late -> data\chunks\late_spans_advanced.parquet


## 12b. Persist semantic chunk embeddings (mapped to data)

The semantic chunker uses ada-002 to *detect boundaries*, but the chunk-level vectors are also directly reusable for retrieval. We embed the final semantic chunks once and persist them mapped 1:1 to `semantic_advanced.parquet` via `chunk_id`.

- Storage: `data/embeddings/semantic_chunk_embeddings.parquet`
- Key: `chunk_id` (joins back to the semantic chunk records)
- A content hash deduplicates identical chunk texts so we never pay to embed the same text twice.
- Vectors come from Azure OpenAI `text-embedding-ada-002` (no Hugging Face / local models).


In [21]:
EMBED_DIR = Path("data/embeddings")
EMBED_DIR.mkdir(parents=True, exist_ok=True)
SEMANTIC_EMB_PATH = EMBED_DIR / "semantic_chunk_embeddings.parquet"

def text_hash(text):
    return hashlib.sha256(normalize_spaces(text).lower().encode()).hexdigest()

if not semantic_available or embedding_client is None:
    print("Azure embeddings unavailable; skipping embedding persistence.")
    semantic_embeddings_df = pd.DataFrame()
elif semantic_df.empty:
    print("No semantic chunks to embed.")
    semantic_embeddings_df = pd.DataFrame()
else:
    emb_df = semantic_df[["chunk_id", "parent_passage_id", "strategy", "text"]].copy()
    emb_df["text_hash"] = emb_df["text"].map(text_hash)

    # Embed each unique text once, then map back to every chunk.
    unique_texts = emb_df.drop_duplicates("text_hash")[["text_hash", "text"]].reset_index(drop=True)
    print(f"Embedding {len(unique_texts)} unique semantic chunk texts (of {len(emb_df)} chunks)...")
    vectors = embed_texts(unique_texts["text"].tolist())
    hash_to_vec = {h: vectors[i] for i, h in enumerate(unique_texts["text_hash"])}

    emb_df["embedding"] = emb_df["text_hash"].map(lambda h: hash_to_vec[h].tolist())
    emb_df["embedding_model"] = SEMANTIC_MODEL_NAME
    emb_df["embedding_dim"] = int(vectors.shape[1])
    semantic_embeddings_df = emb_df[[
        "chunk_id", "parent_passage_id", "strategy", "text_hash",
        "embedding_model", "embedding_dim", "embedding"
    ]]

    # Integrity checks BEFORE persisting: 1 embedding per chunk, dims consistent, ids join back.
    assert len(semantic_embeddings_df) == len(semantic_df), "row count mismatch vs semantic_df"
    assert semantic_embeddings_df["chunk_id"].is_unique, "duplicate chunk_id in embeddings"
    assert set(semantic_embeddings_df["chunk_id"]) == set(semantic_df["chunk_id"]), "chunk_id set mismatch"

    semantic_embeddings_df.to_parquet(SEMANTIC_EMB_PATH, index=False)
    print("Saved:", SEMANTIC_EMB_PATH)
    print("Rows:", len(semantic_embeddings_df), "| dim:", int(vectors.shape[1]),
          "| unique texts embedded:", len(unique_texts))


Embedding 3608 unique semantic chunk texts (of 3610 chunks)...
Saved: data\embeddings\semantic_chunk_embeddings.parquet
Rows: 3610 | dim: 1536 | unique texts embedded: 3608


# 13. Interpretation and handoff

### Semantic
Potentially better topic boundaries, but extra embedding/threshold cost and no guarantee of downstream improvement.

### Biomedical-aware
Protects domain-relevant entity/relation/negation patterns; promising for biomedical retrieval but should be measured per question type.

### Proposition
Excellent retrieval granularity; must be paired with parent/context reconstruction to avoid context loss.

### Parent–child
Separates retrieval precision from generation context.

### Late
Potentially preserves broader contextual information inside chunk embeddings; requires actual long-context embedding experiments.

### What the added measurements (7b, 7c, 8b, 9b, 11c) contribute
- **7b Tokenizer reality check** keeps every size claim honest: whitespace-word budgets are smaller than real model-token budgets.
- **7c Index-cost economics** quantifies chunk explosion (proposition is ~an order of magnitude more chunks) so a localization gain is weighed against embedding/index/rerank cost.
- **8b Biomedical integrity ratios** measure whether entity/relation/negation statements survive intact within a single chunk (split detection vs source), not merely whether a signal is present.
- **9b Semantic cost accounting** records the embedding calls/latency/spend semantic chunking pays to detect boundaries — the input to 'is semantic chunking worth the cost?'.
- **11c Gold-evidence proxy** is a cheap, retriever-free read on whether each strategy keeps answer evidence intact. It is a proxy, **not** a retrieval benchmark.

**No winner is declared here.** These are pre-retrieval diagnostics; the real Recall@K / MRR / faithfulness verdict is produced in Notebook 06/07.

## Next

**Notebook 05 — `05_agentic_chunking`**

will add:

- LLM-guided chunking
- Agentic chunking
- Adaptive/document-aware chunking router
- intrinsic chunk-quality metrics
- chunking-selection accuracy
- chunking regret

Then **Notebook 06** connects all representations to biomedical embeddings, exact dense retrieval and ANN/HNSW.
